# ZhiXia 分阶段测试

**目的**: 逐模块验证重构后的引擎实现，确保与原功能一致。

**测试流程**:
1. 配置加载验证
2. ASR 引擎测试
3. LLM 引擎测试
4. 输出解析器测试
5. RAG retriever 测试
6. TTS 引擎测试
7. Display 接口测试
8. 完整管线端到端测试

In [ ]:
# Cell 1: 配置加载验证

from zhixia.config.settings import AppSettings
from pathlib import Path

# 加载配置
config = AppSettings.load()

print("✅ 配置加载成功")
print(f"\n项目根目录: {config.project_root}")
print(f"LLM 模型路径: {config.llm.model_path}")
print(f"TTS 模型路径: {config.tts.model_path}")
print(f"ASR 引擎: {config.asr.engine}")
print(f"RAG 是否启用: {config.rag.enabled}")
print(f"结构化输出: {config.llm.enable_structured_output}")

# 验证配置文件存在
config_file = config.config_dir / "localconfig.json"
print(f"\n配置文件: {config_file}")
print(f"文件存在: {config_file.exists()}")

In [ ]:
# Cell 2: ASR 引擎测试

import logging
from pathlib import Path

from zhixia.asr.funasr_engine import FunASREngine
from zhixia.asr.whisper_engine import WhisperASREngine
from zhixia.utils.logging import setup_logging

# 设置日志
setup_logging("INFO")

project_root = Path(".")
config = AppSettings.load()

print("测试 FunASR 引擎...")
try:
    asr = FunASREngine(config.asr, project_root)
    print(f"✅ ASR 引擎创建成功: {asr.name}")
    
    # 注意：需要实际的音频文件才能完整测试
    # test_audio = Path(config.asr.input_audio)
    # if test_audio.exists():
    #     result = asr.transcribe(test_audio)
    #     print(f"识别结果: {result.text}")
    # else:
    #     print(f"测试音频不存在: {test_audio}")
except Exception as e:
    print(f"❌ FunASR 测试失败: {e}")

print("\n测试 Whisper 引擎...")
try:
    asr = WhisperASREngine(config.asr)
    print(f"✅ Whisper 引擎创建成功: {asr.name}")
except Exception as e:
    print(f"❌ Whisper 测试失败: {e}")

In [ ]:
# Cell 3: LLM 引擎测试

from zhixia.llm.rkllm_engine import RKLLMEngine
from zhixia.llm.base import LLMMessage

config = AppSettings.load()

print("测试 RKLLM 引擎...")
try:
    llm = RKLLMEngine(config.llm)
    print(f"✅ LLM 引擎创建成功: {llm.name}")
    
    # 设置系统提示
    llm.set_system_prompt("你是一个AI助手。")
    print("✅ 系统提示设置成功")
    
    # 测试简单推理（注意：需要实际模型文件才能运行）
    # messages = [LLMMessage(role="user", content="你好")]
    # response = llm.chat(messages)
    # print(f"回复: {response[:100]}...")
    
except Exception as e:
    print(f"❌ LLM 测试失败: {e}")
    # 注意：如果 librkllmrt.so 不在标准路径，这里会失败

In [ ]:
# Cell 4: 输出解析器测试

from zhixia.llm.output_parser import parse_llm_output, get_format_instruction

print("测试输出解析器...")

test_cases = [
    ('{"text": "你好世界", "emotion": "happy"}', "标准 JSON"),
    ('{"text": "回答", "emotion": "thinking", "detail": "正在思考"}', "带元数据"),
    ('这是普通回复，没有格式', "普通文本"),
    ('<think>让我想想</think最终答案是42', "带思考标签"),
    ('文本{"text": "提取的文本", "emotion": "happy"}更多文本', "JSON 嵌入文本"),
    ('[emotion:sad]今天心情不好', "情绪前缀"),
]

for test_input, desc in test_cases:
    result = parse_llm_output(test_input)
    print(f"\n测试 {desc}:")
    print(f"  输入: {test_input[:50]}...")
    print(f"  输出: text={result.text!r}, emotion={result.emotion!r}")

print("\n格式指令示例:")
print(get_format_instruction()[:200] + "...")

In [ ]:
# Cell 5: RAG retriever 测试

from zhixia.llm.rag.base import RAGRetriever, RAGContext
from zhixia.llm.rag.null_retriever import NullRAGRetriever

print("测试 RAG retriever...")

# 测试 NullRetriever
retriever = NullRAGRetriever()
print(f"✅ Retriever 创建成功: {retriever.name}")

# 测试检索功能
query = "今天天气怎么样？"
context = retriever.retrieve(query, top_k=2)

print(f"\n检索测试 (query: {query}):")
print(f"  结果数量: {len(context.chunks)}")
print(f"  来源描述: {context.source_description}")
for i, chunk in enumerate(context.chunks):
    print(f"  块 {i+1}: {chunk[:50]}...")

# 预留扩展接口
print("\n预留 RAG 引擎接口：")
print("  - zhixia.llm.rag.vector_retriever.VectorRetriever")
print("  - zhixia.llm.rag.keyword_retriever.KeywordRetriever")

In [ ]:
# Cell 6: TTS 引擎测试

from pathlib import Path
from zhixia.config.settings import AppSettings
from zhixia.tts.piper_engine import PiperTTSEngine

config = AppSettings.load()
project_root = Path(".")

print("测试 Piper TTS 引擎...")
try:
    tts = PiperTTSEngine(config.tts, project_root)
    print(f"✅ TTS 引擎创建成功: {tts.name}")
    print(f"✅ 模型可用: {tts.is_available()}")
    
    # 测试合成（注意：需要实际模型文件）
    # test_text = "你好，世界！"
    # output_path = Path("test_output.wav")
    # success = tts.synthesize(test_text, output_path)
    # if success:
    #     print(f"✅ 合成成功: {output_path}")
    # else:
    #     print("❌ 合成失败")
    
except Exception as e:
    print(f"❌ TTS 测试失败: {e}")

In [ ]:
# Cell 7: Display 接口测试

from zhixia.display.base import DisplayPayload
from zhixia.display.null_display import NullDisplay

print("测试 Display 接口...")

display = NullDisplay()
print(f"✅ Display 创建成功: {type(display).__name__}")

# 测试显示内容
payload = DisplayPayload(
    text="你好，我是 ZhiXia！",
    emotion="happy",
    is_thinking=False,
    metadata={"time": "10:30", "user": "test"}
)

display.show(payload)
print("✅ 显示内容已发送")

# 测试思考状态
display.update_thinking(True)
print("✅ 思考状态已激活")

# 测试清除
display.clear()
print("✅ 显示已清除")

print("\n预留 Display 引擎接口：")
print("  - zhixia.display.lcd_display.LCDDisplay")
print("  - zhixia.display.epaper_display.EPaperDisplay")

In [ ]:
# Cell 8: 完整管线端到端测试

from zhixia.pipeline.orchestrator import VoicePipeline
from pathlib import Path

config = AppSettings.load()
project_root = Path(".")

print("准备完整管线测试...")

# 创建各引擎（注意：需要实际模型文件才能运行）
engines = {}

try:
    engines['asr'] = create_asr_engine(config)
    print("✅ ASR 引擎就绪")
except Exception as e:
    print(f"⚠️ ASR 引擎创建失败: {e}")
    
try:
    engines['llm'] = create_llm_engine(config)
    print("✅ LLM 引擎就绪")
except Exception as e:
    print(f"⚠️ LLM 引擎创建失败: {e}")
    
try:
    engines['tts'] = create_tts_engine(config)
    print("✅ TTS 引擎就绪")
except Exception as e:
    print(f"⚠️ TTS 引擎创建失败: {e}")

# 其他组件（无外部依赖）
engines['player'] = ALSAAudioPlayer()
engines['rag'] = create_rag_retriever(config)
engines['display'] = create_display(config)
print("✅ 音频播放器、RAG、Display 就绪")

# 创建管线（即使有引擎失败也创建，用于测试配置加载）
pipeline = VoicePipeline(
    config=config,
    asr_engine=engines['asr'],
    llm_engine=engines['llm'],
    tts_engine=engines['tts'],
    audio_player=engines['player'],
    rag_retriever=engines['rag'],
    display=engines['display'],
)

print("\n✅ 管线创建成功！")
print("\n要运行完整测试，请确保：")
print(f"1. 配置文件中的音频存在: {config.asr.input_audio}")
print(f"2. LLM 模型文件存在: {config.llm.model_path}")
print(f"3. TTS 模型文件存在: {config.tts.model_path}")
print(f"4. librkllmrt.so 可访问（在 rknn_libs/ 目录）")

# 如果所有组件都就绪，可以取消注释运行完整测试
# audio_input = Path(config.asr.input_audio)
# if audio_input.exists():
#     pipeline.process_audio(audio_input)
# else:
#     print(f"\n⚠️ 跳过完整测试：输入音频不存在 {audio_input}")

## 测试说明

每个 cell 可以独立运行，测试对应模块的功能。

**注意事项**：
- 开发机上需要安装依赖才能运行测试
- RK3588 上需要确保模型文件和库文件存在
- 模型加载失败是正常的（文件不存在时）
- 关键是验证接口设计和配置加载正确

**下一步**：
1. 运行这些 cell 验证代码结构
2. 在 RK3588 上运行完整测试
3. 根据测试结果调整实现